# Amazon ML Challenge 2026: Entity Resolution Suite
### Master Google Colab GPU/CPU Runner with Google Drive Persistence & Auto-Resume

Run this notebook in **Google Colab**:
- **Hardware**:  (or high-RAM CPU).

#### Features Engineered for Google Colab:
1. **Drive Directory Structure**: Mirrored after  and  ().
2. **Sharded Data Loading & Split**: Automatic transparent support for compressed  shards so huge files upload quickly to Drive and process with zero memory exhaustion.
3. **Atomic Checkpointing & Auto-Resume ()**: If Colab disconnects or restarts, re-running the cell instantly resumes from the exact last completed batch.
4. **Proactive Memory & OOM Protection**: Real-time RAM/VRAM monitoring with garbage collection and automatic chunk throttling.
5. **Zero-Leakage 3-Tier Scorecard**: Tracks Pair Recall (Blocking), Precision/Recall (Classifier), and Macro F0.5 (Decision Layer) separately.
6. **Official Competition Audit**: Runs  to ensure compliance before saving.

In [ ]:
# 1. Verify GPU and CPU Hardware
!nvidia-smi || echo "Running in CPU Multi-Core Mode"
!lscpu | grep -E "Model name|CPU\(s\):" || true

In [ ]:
# 2. Mount Google Drive and initialize directory structure (mirroring minor-project)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/amazon-ml-challenge-2026")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_OUTPUT = DRIVE_ROOT / "output"

for folder in [DRIVE_ROOT, DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir   : {DRIVE_DATA}")
print(f"    - Weights Dir    : {DRIVE_WEIGHTS}")
print(f"    - Checkpoints Dir: {DRIVE_CHECKPOINTS}")
print(f"    - Output Dir     : {DRIVE_OUTPUT}")

In [ ]:
# 3. Clone or pull the repository
# If cloning from GitHub, set your REPO_URL below:
import os
REPO_URL = "https://github.com/YOUR_USERNAME/amazon-ml-challenge.git"
REPO_DIR = "/content/amazon-ml-challenge"

if not os.path.exists(REPO_DIR):
    if os.path.exists("/content/drive/MyDrive/amazon-ml-challenge-2026/repo"):
        !cp -r /content/drive/MyDrive/amazon-ml-challenge-2026/repo 
    else:
        !git clone   || mkdir -p 
%cd 
if os.path.exists(".git"):
    !git pull || true

In [ ]:
# 4. Install Dependencies (Prefers uv, fallback to pip)
!python install.py

In [ ]:
# 5. Dataset Symlinking & Sharding Check
# Link Drive dataset into local data/ directory if present on Drive
import os
from pathlib import Path

!mkdir -p data
if DRIVE_DATA.exists():
    for item in DRIVE_DATA.iterdir():
        local_target = Path(f"data/{item.name}")
        if not local_target.exists():
            os.symlink(item, local_target)
            print(f"[*] Linked Drive data: {item.name} -> data/{item.name}")

# Verify available dataset paths (supports both sharded and raw TSV formats)
!ls -la data/ || true

In [ ]:
# 6. Fast Smoke Test (~10 seconds)
# Validates normalization, blocking, classifier, decision layer & audit end-to-end
!python scripts/run_pipeline.py \n    --train-dir dataset_split/train \n    --test-dir dataset_split/test \n    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \n    --smoke-test

In [ ]:
# 7. Zero-Leakage 5-Fold Cross-Validation
# Tracks Blocking Recall, Classifier Precision/Recall, and Decision Layer Macro F0.5
!python src/validation.py \n    --mode cv \n    --train-dir dataset_split/train \n    --n-splits 5

In [ ]:
# 8. Full Benchmark Training & Submission Generation (Crash-Resilient with Auto-Resume)
# If Colab restarts or disconnects, re-run this cell to resume without losing work!
!python scripts/run_pipeline.py \n    --train-dir dataset_split/train \n    --test-dir dataset_split/test \n    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \n    --device auto \n    --batch-size 2000

In [ ]:
# 9. Run Official Submission Validator on Output Artifacts
!python 6ab10eb3b23ba_student_resource/student_resource/utils/validate_submission.py \n    --matching output/matching_results.tsv \n    --candidate output/candidate_pairs.tsv \n    --test-dir dataset_split/test \n    --check-ids

In [ ]:
# 10. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_path = Path("output/scorecard.json")
if scorecard_path.is_file():
    sc = json.loads(scorecard_path.read_text())
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get("blocking_pair_recall", 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get("candidate_reduction_rate", 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get("candidates_per_entity", 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get("classifier_pair_precision", 0)*100:.2f}%** |
| | Pair Recall | **{sc.get("classifier_pair_recall", 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get("classifier_pair_f1", 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get("official_macro_f05", 0):.4f}** |
| | Singleton Sub-Score | **{sc.get("singleton_macro_f05", 0):.4f}** |
| | Matched Sub-Score | **{sc.get("matched_macro_f05", 0):.4f}** |
"""
    display(Markdown(md_table))

print("
--- First 5 rows of candidate_pairs.tsv ---")
!head -n 6 output/candidate_pairs.tsv
print("
--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/matching_results.tsv